# DATA INTENSIVE SYSTEMS IN BIOMEDICINE

## Opis projektu: ....

Tu się kiedyś coś dopisze

---
## Wymogi: 
* ``Przeprowadź czyszczenie danych (jesli jest to wymagane lub nie – w obu przypadkach uzasadnij), zweryfikuj ilość pustych wartości i czy mają jakiś wpływ na analizę ``
* ``Przedstaw klasyfikację danych ``
* ``Wykorzystaj MLlib do predykcji ``
* ``Uwzględnij wizualizację danych za pomocą bibliotek pandas lub matplotlib.``
* ``Skorzystaj z paradygmatu MapReduce oraz funkcji lambda. ``
---

## Wczytanie danych za pomocą spark i wyświetlenie 10 pierwszych wierszy

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Diabetes Analysis") \
    .getOrCreate()

DATA_PATH = "/app/data/diabetes_health_indicators.csv"

df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

df.show(10, truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 19:18:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|Diabetes_012|HighBP|HighChol|CholCheck|BMI |Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex|Age |Education|Income|
+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|0.0         |1.0   |1.0     |1.0      |40.0|1.0   |0.0   |0.0                 |0.0         |0.0   |1.0    |0.0              |1.0          |0.0        |5.0    |18.0    |15.0    |1.0     |0.0|9.0 |4.0      |3.0   |
|0.0         |0.0   |0.0     |0.0      |25.0|1.0   |0.0   |0.0                 |1.0         |0.0   |0.0    |0.0              |0.0          |1.0 

In [27]:
print("-" * 15 + "Informacje o Strukturze bazy danych" + "-" * 15)
print(f'Całkowita ilość wierszy: {df.count()}')
print(f'Całkowita ilość kolumn: {len(df.columns)}')
print(f'Nazwy kolumn: {df.columns}')
print(65 * "-")

---------------Informacje o Strukturze bazy danych---------------


Całkowita ilość wierszy: 253680
Całkowita ilość kolumn: 22
Nazwy kolumn: ['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
-----------------------------------------------------------------


## Sprawdzenie czy występują puste wiersze

In [ ]:
from pyspark.sql.functions import col, count, when

# Zliczanie brakujących (null) wartości w każdej kolumnie
missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df.columns
])

missing_values.show(truncate=False)

+------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+---+---------+------+
|Diabetes_012|HighBP|HighChol|CholCheck|BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex|Age|Education|Income|
+------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+---+---------+------+
|0           |0     |0       |0        |0  |0     |0     |0                   |0           |0     |0      |0                |0            |0          |0      |0       |0       |0       |0  |0  |0        |0     |
+------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-

Dla każdej kolumny sprawdzono liczbę wartości NULL. Wyniki pokazują, że:

- żadna kolumna nie zawiera brakujących danych,
- wszystkie wartości wynoszą 0 (brak braków).

W związku z tym zbiór danych nie wymagał procesu czyszczenia. Brak brakujących danych oznacza, że dane są kompletne i mogą zostać bezpośrednio wykorzystane do dalszej analizy oraz budowy modelu ML.

---

## Przedstawienie klasyfikacji danych

In [28]:
df.printSchema()

root
 |-- Diabetes_012: double (nullable = true)
 |-- HighBP: double (nullable = true)
 |-- HighChol: double (nullable = true)
 |-- CholCheck: double (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: double (nullable = true)
 |-- Stroke: double (nullable = true)
 |-- HeartDiseaseorAttack: double (nullable = true)
 |-- PhysActivity: double (nullable = true)
 |-- Fruits: double (nullable = true)
 |-- Veggies: double (nullable = true)
 |-- HvyAlcoholConsump: double (nullable = true)
 |-- AnyHealthcare: double (nullable = true)
 |-- NoDocbcCost: double (nullable = true)
 |-- GenHlth: double (nullable = true)
 |-- MentHlth: double (nullable = true)
 |-- PhysHlth: double (nullable = true)
 |-- DiffWalk: double (nullable = true)
 |-- Sex: double (nullable = true)
 |-- Age: double (nullable = true)
 |-- Education: double (nullable = true)
 |-- Income: double (nullable = true)



### Opis struktury danych

Dzięki zastosowaniu funkcji `printSchema()` możliwe jest wyświetlenie struktury analizowanej bazy danych. Analizując wynik funkcji można zauważyć, że:
- wszystkie kolumny w zbiorze danych są typu numerycznego (`double`),
- każda kolumna dopuszcza wartości NULL (nullable = true), jednak wcześniejsza analiza wykazała brak brakujących danych.

In [29]:
df.describe().show(truncate=False)

+-------+-------------------+------------------+-------------------+------------------+------------------+-------------------+-------------------+--------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+-----------------+------------------+------------------+
|summary|Diabetes_012       |HighBP            |HighChol           |CholCheck         |BMI               |Smoker             |Stroke             |HeartDiseaseorAttack|PhysActivity      |Fruits            |Veggies            |HvyAlcoholConsump  |AnyHealthcare      |NoDocbcCost        |GenHlth           |MentHlth          |PhysHlth          |DiffWalk           |Sex                |Age              |Education         |Income            |
+-------+-------------------+------------------+-------------------+------------------+------------------+----------------

### Podział zmiennych

Na podstawie analizy zbioru danych można wyróżnić kilka typów zmiennych, które się w nim znajdują:


### 1. Zmienna docelowa (target)

- **Diabetes_012**:
  - 0 – brak cukrzycy
  - 1 – stan przedcukrzycowy
  - 2 – cukrzyca

Jest to zmienna, którą model ML będzie przewidywał.

---

### 2. Zmienne binarne (0/1)

Reprezentują występowanie lub brak danego czynnika:

- HighBP – nadciśnienie  
- HighChol – wysoki cholesterol  
- CholCheck – badanie cholesterolu  
- Smoker – palenie  
- Stroke – udar  
- HeartDiseaseorAttack – choroby serca  
- PhysActivity – aktywność fizyczna  
- Fruits – spożycie owoców  
- Veggies – spożycie warzyw  
- HvyAlcoholConsump – nadużywanie alkoholu  
- AnyHealthcare – dostęp do opieki zdrowotnej  
- NoDocbcCost – brak wizyty lekarskiej z powodu kosztów  
- DiffWalk – trudności z poruszaniem się  
- Sex – płeć  

---

### 3. Zmienne ciągłe (ilościowe)

- **BMI** – wskaźnik masy ciała (zakres: 12–98)


---

### 4. Zmienne kategoryczne

Zmienne posiadają ograniczoną liczbę wartości i reprezentują kategorie uporządkowane:

- **GenHlth** (1–5) – ogólny stan zdrowia  
- **MentHlth** (0–30) – liczba dni problemów psychicznych  
- **PhysHlth** (0–30) – liczba dni problemów fizycznych  
- **Age** (1–13) – grupy wiekowe  
- **Education** (1–6) – poziom edukacji  
- **Income** (1–8) – poziom dochodów  

---

Na podstawie wcześniejszej analizy danych możliwe było określenie typów zmiennych:
- zmienne binarne,
- zmienne kategoryczne (porządkowe),
- zmienne ciągłe.

Dla zmiennych **binarnych i kategorycznych** zastosowano analizę rozkładu wartości. Żeby zapoznać się z rozkładem tego typu danych.

In [33]:
categorical_columns = [c for c in df.columns if c != "BMI"]

total_count = df.count()

for col_name in categorical_columns:
    print(f"\nKolumna: {col_name}")
    
    df.groupBy(col_name) \
      .count() \
      .withColumn("percentage", (col("count") / total_count) * 100) \
      .orderBy(col_name) \
      .show(truncate=False)


Kolumna: Diabetes_012


+------------+------+------------------+
|Diabetes_012|count |percentage        |
+------------+------+------------------+
|0.0         |213703|84.24116997792494 |
|1.0         |4631  |1.8255282245348472|
|2.0         |35346 |13.933301797540206|
+------------+------+------------------+


Kolumna: HighBP


+------+------+------------------+
|HighBP|count |percentage        |
+------+------+------------------+
|0.0   |144851|57.099889624724064|
|1.0   |108829|42.90011037527594 |
+------+------+------------------+


Kolumna: HighChol


+--------+------+-----------------+
|HighChol|count |percentage       |
+--------+------+-----------------+
|0.0     |146089|57.58790602333649|
|1.0     |107591|42.41209397666351|
+--------+------+-----------------+


Kolumna: CholCheck


+---------+------+------------------+
|CholCheck|count |percentage        |
+---------+------+------------------+
|0.0      |9470  |3.7330495111952064|
|1.0      |244210|96.2669504888048  |
+---------+------+------------------+


Kolumna: Smoker


+------+------+-----------------+
|Smoker|count |percentage       |
+------+------+-----------------+
|0.0   |141257|55.68314411857458|
|1.0   |112423|44.31685588142542|
+------+------+-----------------+


Kolumna: Stroke


... (7 outputs left out)

+--------+------+--------------------+
|MentHlth|count |percentage          |
+--------+------+--------------------+
|0.0     |175680|69.25260170293282   |
|1.0     |8538  |3.36565752128666    |
|2.0     |13054 |5.145853043204037   |
|3.0     |7381  |2.9095711132134974  |
|4.0     |3789  |1.4936140018921475  |
|5.0     |9030  |3.5596026490066226  |
|6.0     |988   |0.3894670450961842  |
|7.0     |3100  |1.2220119836013876  |
|8.0     |639   |0.2518921475875118  |
|9.0     |91    |0.0358719646799117  |
|10.0    |6373  |2.5122201198360137  |
|11.0    |41    |0.016162093976663513|
|12.0    |398   |0.15689057079785557 |
|13.0    |41    |0.016162093976663513|
|14.0    |1167  |0.4600283822138127  |
|15.0    |5505  |2.1700567644276254  |
|16.0    |88    |0.03468937243771681 |
|17.0    |54    |0.02128666035950804 |
|18.0    |97    |0.03823714916430148 |
|19.0    |16    |0.00630715862503942 |
+--------+------+--------------------+
only showing top 20 rows


Kolumna: PhysHlth


+--------+------+--------------------+
|PhysHlth|count |percentage          |
+--------+------+--------------------+
|0.0     |160052|63.09208451592557   |
|1.0     |11388 |4.489120151371806   |
|2.0     |14764 |5.819930621255125   |
|3.0     |8495  |3.3487070324818666  |
|4.0     |4542  |1.790444654683065   |
|5.0     |7622  |3.0045726900031537  |
|6.0     |1330  |0.5242825607064018  |
|7.0     |4538  |1.7888678650268053  |
|8.0     |809   |0.31890570797855566 |
|9.0     |179   |0.07056133711762852 |
|10.0    |5595  |2.205534531693472   |
|11.0    |60    |0.023651844843897825|
|12.0    |578   |0.22784610532954902 |
|13.0    |68    |0.026805424156417533|
|14.0    |2587  |1.0197887101860612  |
|15.0    |4916  |1.9378744875433618  |
|16.0    |112   |0.04415011037527594 |
|17.0    |96    |0.03784295175023652 |
|18.0    |152   |0.05991800693787449 |
|19.0    |22    |0.008672343109429203|
+--------+------+--------------------+
only showing top 20 rows


Kolumna: DiffWalk
+--------+------+--

+----+-----+------------------+
|Age |count|percentage        |
+----+-----+------------------+
|1.0 |5700 |2.2469252601702934|
|2.0 |7598 |2.9951119520655944|
|3.0 |11123|4.384657836644592 |
|4.0 |13823|5.448990854619994 |
|5.0 |16157|6.369047619047619 |
|6.0 |19819|7.812598549353517 |
|7.0 |26314|10.372910753705455|
|8.0 |30832|12.153894670450962|
|9.0 |33244|13.104698833175654|
|10.0|32194|12.690791548407443|
|11.0|23533|9.276647745190791 |
|12.0|15980|6.299274676758121 |
|13.0|17363|6.844449700409966 |
+----+-----+------------------+


Kolumna: Education


+---------+------+-------------------+
|Education|count |percentage         |
+---------+------+-------------------+
|1.0      |174   |0.06859035004730368|
|2.0      |4043  |1.5937401450646482 |
|3.0      |9478  |3.736203090507726  |
|4.0      |62750 |24.735887732576476 |
|5.0      |69910 |27.558341217281612 |
|6.0      |107325|42.30723746452224  |
+---------+------+-------------------+


Kolumna: Income
+------+-----+------------------+
|Income|count|percentage        |
+------+-----+------------------+
|1.0   |9811 |3.867470829391359 |
|2.0   |11783|4.644828129927467 |
|3.0   |15994|6.30479344055503  |
|4.0   |20135|7.937164932198044 |
|5.0   |25883|10.203011668243455|
|6.0   |36470|14.376379690949229|
|7.0   |43219|17.036818038473665|
|8.0   |90385|35.629533270261746|
+------+-----+------------------+

